# 第 3 章实践：RMSNorm

RMSNorm 是大模型中最常见的归一化算子之一，计算链同时包含归约与逐元素模式，是检验本章学习成果的最佳实践。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）`pl.sum(out, src, tmp, dim=0)` 对 [8, 256] 源 Tile 归约，输出 shape = ______；随后若要恢复 [8, 256] 需做 ______。
2. （选择）Tile 计算接口的参数约定是：
   A. 输出在第一个参数<br/>B. 输出在最后一个参数<br/>C. 通过返回值返回
3. （判断）vf.* 操作可以直接写在 `pl.section_vector()` 内，不必定义 VF 函数。（对/错）
4. （选择）以下哪个操作不能出现在 VF 函数体内？
   A. `vf.mul(reg_a, reg_b, preg)`<br/>B. `pl.range(0, 8)`<br/>C. `buf_group.next()`

In [ ]:
!cat ./answer/03.05_objective_answer.txt

## 2. 编程实践一（简单）：逐元素三件套

**任务**：实现 `out = relu(x) * alpha + beta`（[8, 256] FP32），只用 Tile 接口，列出调用顺序。解释为什么不能直接用 add_relu 替代这里的 ReLU 与最后一次加法；另将公式改为 `relu(x+b)*alpha`（b 为同形 Tile），比较独立调用与 add_relu 的接口数量。


In [ ]:
# 在此完成实践一
pass

## 3. 编程实践二（中等）：行内 RMSNorm（Tile 版）

**任务**：实现 RMSNorm：`y = x / sqrt(mean(x²) + eps)`，按行归一化，x 为 [8, 256] FP32，eps=1e-5。本练习省略可学习的 gamma，仅实现 RMS 归一化核心。

**提示（推导路线）**：

1. `tmp = x * x`（Tile-Tile 逐元素）；
2. `ms = sum(tmp, dim=0)`（每行的平方和，输出 [8,1]）；
3. `ms = ms / N + eps`（Tile-Scalar；N=256）；
4. `rstd = 1 / sqrt(ms)`（查 API：rsqrt 或 sqrt+div 组合）；
5. `y = x * rstd`（广播：[8,256] × [8,1] 的按行缩放，用 `pl.expand_mul(y_tile, x_tile, rstd_tile, dim=0)`；[8,1] Tile 不等于 Host/Device 标量）。


In [ ]:
# 在此完成实践二
pass

## 4. 编程实践三（困难）：RMSNorm 的 VF 改写

**任务**：把实践二中"平方和归约"到"逐元素归一化"的连续段改写为 VF 函数，寄存器内完成"load x → 平方 → 归约累加 → rstd → 归一化 → store"，按寄存器片段让 x 只从 UB 读一次、y 只写一次，并评估寄存器占用。

**提示**：查 vf.reduce_* 家族的接口签名；掩码用 `vf.create_mask(pattern=pl.MaskPattern.ALL, ...)` 创建；FP32 每个寄存器 64 个元素，一行 256 个元素需 4 个片段；reduce_sum 的结果在 lane 0。片段累加、除以 256、加 eps、rsqrt，以及将 lane 0 广播到全部 lane，都是必须实现的步骤。


In [ ]:
# 在此完成实践三
pass

In [ ]:
!cat ./answer/03.05_practice_answer.txt

### 完整参考实现与验收

三个题目对应 `relu_affine`、`rms_tile`、`rms_vf`。VF 的四个 x 寄存器保留到最终乘法；`vf.full` 广播归约 lane 0。校验覆盖随机值、零、负常量和小幅值，重点观察 eps 与除 N 是否正确。

[参考源码](./src/rmsnorm_examples.py) 可在 950 上从章节目录执行 `python src/rmsnorm_examples.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/rmsnorm_examples.py


## 5. 本节小结

RMSNorm 综合了**归约、广播、Tile-Scalar、可选的 VF 改写**——这条"先用 Tile 跑通、再把热点段进寄存器"的路径，就是 Pro 矢量算子开发的标准节奏。第 4 章把战场切到矩阵：Cube 计算模型、分形布局与多核切分。